In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# =====================================
# DATASET PATH
# =====================================

dataset_path = "/kaggle/input/datasets/sabbir4724/sabbirs-dataset"

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        [0.485,0.456,0.406],
        [0.229,0.224,0.225]
    )
])

# Load dataset
full_dataset = datasets.ImageFolder(
    dataset_path,
    transform=transform
)

# Split exactly like your original code
total = len(full_dataset)

train_size = int(0.7*total)
val_size = int(0.15*total)
test_size = total-train_size-val_size

train_set, val_set, test_set = random_split(
    full_dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

# Create test_loader
test_loader = DataLoader(
    test_set,
    batch_size=64,
    shuffle=False,
    num_workers=2
)

print("Test samples:", len(test_set))

In [ ]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torchvision import models
from sklearn.manifold import TSNE

# =====================================
# PATHS
# =====================================

model_path = "/kaggle/input/datasets/sabbir4724/modelpath/best_model.pth"

output_dir = "/kaggle/working/output"
os.makedirs(output_dir, exist_ok=True)

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

class_names = [
    "Normal",
    "Pneumonia Bacterial",
    "Pneumonia Viral"
]

num_classes = len(class_names)


# =====================================
# SE BLOCK
# =====================================

class SEBlock(nn.Module):

    def __init__(self, channels, reduction=16):
        super().__init__()

        self.fc1 = nn.Linear(
            channels,
            channels//reduction
        )

        self.fc2 = nn.Linear(
            channels//reduction,
            channels
        )

        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self,x):

        b,c,_,_=x.size()

        y=x.mean(dim=(2,3))

        y=self.fc1(y)
        y=self.relu(y)
        y=self.fc2(y)

        y=self.sigmoid(y).view(
            b,c,1,1
        )

        return x*y.expand_as(x)


# =====================================
# CBAM
# =====================================

class CBAM(nn.Module):

    def __init__(
        self,
        channels,
        reduction=16,
        kernel_size=7
    ):

        super().__init__()

        self.mlp=nn.Sequential(
            nn.Linear(
                channels,
                channels//reduction
            ),
            nn.ReLU(),
            nn.Linear(
                channels//reduction,
                channels
            )
        )

        self.conv=nn.Conv2d(
            2,
            1,
            kernel_size,
            padding=kernel_size//2
        )

        self.sigmoid=nn.Sigmoid()

    def forward(self,x):

        avg_pool=x.mean(dim=(2,3))
        max_pool=torch.amax(x,dim=(2,3))

        ca=self.mlp(avg_pool)+self.mlp(max_pool)

        ca=torch.sigmoid(ca).view(
            x.size(0),
            x.size(1),
            1,
            1
        )

        x=x*ca

        avg_out=x.mean(
            dim=1,
            keepdim=True
        )

        max_out=torch.amax(
            x,
            dim=1,
            keepdim=True
        )

        sa=torch.cat(
            [avg_out,max_out],
            dim=1
        )

        sa=self.sigmoid(
            self.conv(sa)
        )

        x=x*sa

        return x


# =====================================
# Unfreeze helper
# =====================================

def unfreeze_last_n_params(
    model,
    n
):

    for p in model.parameters():
        p.requires_grad=False

    params=list(
        model.parameters()
    )

    for p in params[-n:]:
        p.requires_grad=True


# =====================================
# Fusion Model
# =====================================

class FusionModel(nn.Module):

    def __init__(
        self,
        num_classes
    ):

        super().__init__()

        self.densenet=models.densenet121(
            pretrained=False
        )

        self.densenet.classifier=nn.Identity()

        self.se_d=SEBlock(1024)

        self.head_d=nn.Sequential(
            nn.Linear(1024,512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.ReLU(),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        self.efficientnet=models.efficientnet_b0(
            pretrained=False
        )

        self.efficientnet.classifier=nn.Identity()

        self.se_e=SEBlock(1280)

        self.head_e=nn.Sequential(
            nn.Linear(1280,512),
            nn.BatchNorm1d(512),
            nn.Dropout(0.3),
            nn.ReLU(),

            nn.Linear(512,256),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3),
            nn.ReLU()
        )

        fusion_dim=512

        self.cbam=CBAM(
            fusion_dim
        )

        self.classifier=nn.Sequential(
            nn.Linear(
                fusion_dim,
                128
            ),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(
                128,
                num_classes
            )
        )


# =====================================
# Load model
# =====================================

model=FusionModel(
    num_classes
).to(device)

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device
    )
)

model.eval()

print("Model loaded")


# =====================================
# t-SNE extraction
# Requires existing test_loader
# =====================================

features=[]
predictions=[]

with torch.no_grad():

    for X,y in test_loader:

        X=X.to(device)

        f_d=model.densenet(X)
        f_d=f_d.unsqueeze(-1).unsqueeze(-1)
        f_d=model.se_d(f_d)
        f_d=f_d.squeeze(-1).squeeze(-1)
        f_d=model.head_d(f_d)

        f_e=model.efficientnet(X)
        f_e=f_e.unsqueeze(-1).unsqueeze(-1)
        f_e=model.se_e(f_e)
        f_e=f_e.squeeze(-1).squeeze(-1)
        f_e=model.head_e(f_e)

        f=torch.cat(
            [f_d,f_e],
            dim=1
        )

        f=f.unsqueeze(-1).unsqueeze(-1)
        f=model.cbam(f)
        f=f.squeeze(-1).squeeze(-1)

        outputs=model.classifier(f)

        pred=outputs.argmax(
            dim=1
        )

        features.append(
            f.cpu().numpy()
        )

        predictions.extend(
            pred.cpu().numpy()
        )


features=np.concatenate(
    features
)

predictions=np.array(
    predictions
)

tsne=TSNE(
    n_components=2,
    random_state=42,
    perplexity=30
)

result=tsne.fit_transform(
    features
)

plt.figure(figsize=(12,8))

for i,name in enumerate(class_names):

    idx=predictions==i

    plt.scatter(
        result[idx,0],
        result[idx,1],
        label=name,
        alpha=0.7
    )

plt.legend()

plt.title(
    "t-SNE Predicted Class Analysis"
)

plt.savefig(
    f"{output_dir}/tsne_predicted.png",
    dpi=300
)

plt.show()

print(
    "Saved:",
    f"{output_dir}/tsne_predicted.png"
)